# 04 · Stage 2b — malignant subtype {mel, bcc, akiec} (multimodal)

Given a malignant decision at Stage 1, classify the malignant subtype. `akiec` is the rare class the diffusion ablation (notebook 05) targets.

**Design.** We train **two** models on the *same* EfficientNetB3 backbone so any
performance gap is attributable to the metadata, not a different vision model:

1. **Image-only baseline** — the original project's architecture (the control).
2. **Multimodal late-fusion** — image embedding ⊕ MLP over age/sex/site.

Both are trained on the **full** stage data with **balanced class weights** and
on-the-fly augmentation — no undersampling (the key methodological fix). Models
are saved to `models/` with names the inference `pipeline.py` expects.

---
*Part of: **A Multimodal, Explainable Hierarchical Framework for Skin-Lesion
Classification with Diffusion-Based Augmentation of Rare Malignancies.***
All training/eval logic lives in the tested modules under `src/`; these
notebooks orchestrate them on a Colab T4.

In [1]:
!pip -q install scikit-learn matplotlib kaggle >/dev/null 2>&1
print("deps ready")

deps ready


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import shutil

DRIVE_PROJECT = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'
#
LOCAL_PROJECT = '/content/Skin_Cancer_Research_Grant2024'

os.makedirs(LOCAL_PROJECT, exist_ok=True)

folders = ['data']

for folder in folders:
    source = os.path.join(DRIVE_PROJECT, folder)
    destination = os.path.join(LOCAL_PROJECT, folder)

    print(f"Copying {folder}...")

    shutil.copytree(
        source,
        destination,
        dirs_exist_ok=True
    )

    print(f"✓ {folder} copied")

print("\nAll folders copied to Colab!")

Copying data...
✓ data copied

All folders copied to Colab!


In [8]:
import os, sys

try:
    PROJECT_DIR = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'
except Exception:
    PROJECT_DIR = os.path.abspath('..')

SRC       = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR= os.path.join(PROJECT_DIR, 'models')
DATA_DIR  = '/content/Skin_Cancer_Research_Grant2024/data'
FIG_DIR   = os.path.join(PROJECT_DIR, 'report', 'figures')
SPLIT_CSV = os.path.join(MODELS_DIR, 'ham_split.csv')
PREP_JSON = os.path.join(MODELS_DIR, 'preprocessing.json')
for d in (MODELS_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.isdir(SRC), f"src/ not found at {SRC} — set PROJECT_DIR correctly"
sys.path.insert(0, SRC)
print("PROJECT_DIR =", PROJECT_DIR)

PROJECT_DIR = /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024


In [9]:
import os, json
import pandas as pd
import data as D, metadata as M, model as Model

STAGE = "malignant"
EPOCHS = 30
BATCH = 32

proc = M.MetadataProcessor.load(PREP_JSON)
df = pd.read_csv(SPLIT_CSV)
sdf = D.filter_for_stage(df, STAGE)
tr = sdf[sdf.split == 'train']; va = sdf[sdf.split == 'val']; te = sdf[sdf.split == 'test']
n_classes = len(D.STAGE_CLASSES[STAGE])
class_weight = D.compute_class_weights(tr, STAGE)
print("classes:", D.STAGE_CLASSES[STAGE], "| n_classes:", n_classes)
print("train/val/test:", len(tr), len(va), len(te))
print("class weights:", class_weight)

classes: ['mel', 'bcc', 'akiec'] | n_classes: 3
train/val/test: 1412 249 293
class weights: {0: 0.5854063018242123, 1: 1.2652329749103943, 2: 1.9943502824858756}


In [10]:
# ---------------- Image-only baseline (control) ----------------
tf_tr = D.make_dataset(tr, STAGE, None, multimodal=False, batch_size=BATCH, training=True, augment=True)
tf_va = D.make_dataset(va, STAGE, None, multimodal=False, batch_size=BATCH, training=False)

baseline = Model.compile_model(Model.build_image_only_baseline(n_classes))
base_path = os.path.join(MODELS_DIR, "malignant_baseline.keras")
hist_b = baseline.fit(tf_tr, validation_data=tf_va, epochs=EPOCHS,
                      class_weight=class_weight,
                      callbacks=Model.make_callbacks(base_path))
baseline.save(base_path)
json.dump(hist_b.history, open(base_path.replace('.keras', '_history.json'), 'w'))
print("saved baseline ->", base_path)

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 307s 3s/step - accuracy: 0.4759 - loss: 1.3288 - val_accuracy: 0.5823 - val_loss: 1.6197 - learning_rate: 1.0000e-04
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 42s 908ms/step - accuracy: 0.5453 - loss: 1.0903 - val_accuracy: 0.6305 - val_loss: 1.1202 - learning_rate: 1.0000e-04
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.5928 - loss: 0.9930 - val_accuracy: 0.6627 - val_loss: 0.9284 - learning_rate: 1.0000e-04
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.6508 - loss: 0.9021 - val_accuracy: 0.6988 - val_loss: 0.7922 - learning_rate: 1.0000e-04
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 77s 934ms/step - accuracy: 0.6480 - loss: 0.8644 - val_accuracy: 0.7229 - val_loss: 0.7205 - learning_rate: 1.0000e-04
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.6537 - loss: 0.8750 - val_accuracy: 0.7149 - val_loss: 0.6821 - learning_rate: 5.4881e-05
Epoch 7/30
45/45 ━

In [11]:
# ---------------- Multimodal late-fusion ----------------
tf_tr_m = D.make_dataset(tr, STAGE, proc, multimodal=True, batch_size=BATCH, training=True, augment=True)
tf_va_m = D.make_dataset(va, STAGE, proc, multimodal=True, batch_size=BATCH, training=False)

mm = Model.compile_model(Model.build_multimodal_model(n_classes, proc.feature_dim))
mm_path = os.path.join(MODELS_DIR, "malignant_multimodal.keras")
hist_m = mm.fit(tf_tr_m, validation_data=tf_va_m, epochs=EPOCHS,
                class_weight=class_weight,
                callbacks=Model.make_callbacks(mm_path))
mm.save(mm_path)
json.dump(hist_m.history, open(mm_path.replace('.keras', '_history.json'), 'w'))
print("saved multimodal ->", mm_path)

Epoch 1/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 264s 3s/step - accuracy: 0.3846 - loss: 1.3165 - val_accuracy: 0.5261 - val_loss: 1.2449 - learning_rate: 1.0000e-04
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 96s 917ms/step - accuracy: 0.4809 - loss: 1.1133 - val_accuracy: 0.5663 - val_loss: 1.0774 - learning_rate: 1.0000e-04
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 44s 984ms/step - accuracy: 0.5071 - loss: 1.1040 - val_accuracy: 0.6185 - val_loss: 0.9215 - learning_rate: 1.0000e-04
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.5567 - loss: 1.0106 - val_accuracy: 0.6506 - val_loss: 0.8112 - learning_rate: 1.0000e-04
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.5751 - loss: 0.9740 - val_accuracy: 0.6386 - val_loss: 0.7821 - learning_rate: 1.0000e-04
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 44s 974ms/step - accuracy: 0.5701 - loss: 0.9809 - val_accuracy: 0.6787 - val_loss: 0.7904 - learning_rate: 5.4881e-05
Epoch 7/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 41s 899ms/step - accuracy: 0

In [12]:
# Quick val-accuracy comparison.
print("baseline   best val_acc: %.4f" % max(hist_b.history['val_accuracy']))
print("multimodal best val_acc: %.4f" % max(hist_m.history['val_accuracy']))

baseline   best val_acc: 0.7229
multimodal best val_acc: 0.6787
